In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress

# Model parameters
N = 1e5
i0 = 10.0
R0 = 1.5

def f(t, i):
    """RHS of di/dt = (R0-1) i - R0 i^2 / N"""
    return (R0 - 1) * i - R0 * i**2 / N

def i_exact(t, t0=0.0, i_t0=i0):
    """Closed-form solution of the SIS ODE, started from i(t0) = i_t0."""
    K = N * (1 - 1 / R0)
    tau = t - t0
    return K * i_t0 * np.exp((R0 - 1) * tau) / (K + i_t0 * (np.exp((R0 - 1) * tau) - 1))

# Single-step functions
def euler_step(f, t, y, h):
    return y + h * f(t, y)

def rk2_step(f, t, y, h):
    k1 = f(t, y)
    k2 = f(t + h, y + h * k1)
    return y + h / 2 * (k1 + k2)

def rk4_step(f, t, y, h):
    k1 = f(t, y)
    k2 = f(t + h / 2, y + h / 2 * k1)
    k3 = f(t + h / 2, y + h / 2 * k2)
    k4 = f(t + h, y + h * k3)
    return y + h / 6 * (k1 + 2 * k2 + 2 * k3 + k4)

STEPPERS = {"Euler": euler_step, "RK2": rk2_step, "RK4": rk4_step}

# Generic integrator (used for part c/global error)
def integrate(step_fn, f, y0, T, M):
    h = T / M
    t = np.linspace(0, T, M + 1)
    y = np.empty(M + 1)
    y[0] = y0
    for j in range(M):
        y[j + 1] = step_fn(f, t[j], y[j], h)
    return t, y

# M values: log-spaced integers from 100 to 2000: h = 20/M in [0.01, 0.2]
Ms = np.round(np.logspace(np.log10(100), np.log10(2000), 10)).astype(int)
hs = 20.0 / Ms

# Question 1.1 (b)
t0_values = [2, 10, 20]

local_results = {t0: {name: np.empty(len(hs)) for name in STEPPERS} for t0 in t0_values}

for t0 in t0_values:
    y_t0 = i_exact(t0)  # exact value to start the single step from
    for k, h in enumerate(hs):
        y_true_next = i_exact(t0 + h, t0=t0, i_t0=y_t0)  # exact value at t0+h
        for name, step_fn in STEPPERS.items():
            y_num_next = step_fn(f, t0, y_t0, h)
            local_results[t0][name][k] = abs(y_num_next - y_true_next)

# fit slopes
local_slopes = {t0: {} for t0 in t0_values}
for t0 in t0_values:
    for name in STEPPERS:
        eps = local_results[t0][name]
        res = linregress(np.log(hs), np.log(eps))
        local_slopes[t0][name] = (res.slope, res.intercept, res.rvalue**2)

# plot
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=False)
colors = {"Euler": "tab:blue", "RK2": "tab:orange", "RK4": "tab:green"}
for ax, t0 in zip(axes, t0_values):
    for name in STEPPERS:
        eps = local_results[t0][name]
        slope, intercept, r2 = local_slopes[t0][name]
        ax.loglog(hs, eps, "o-", color=colors[name],
                   label=f"{name} (slope={slope:.2f})")
    ax.set_xlabel("h")
    ax.set_title(f"Local error, $t_0={t0}$")
    ax.grid(True, which="both", alpha=0.3)
    ax.legend()
axes[0].set_ylabel(r"$\epsilon = |i_{num} - i_{exact}(t_0+h)|$")
plt.tight_layout()
plt.savefig("local_error.png", dpi=150)
plt.show()

print("Question 1.1(b): Local error slopes")
for t0 in t0_values:
    print(f"t0 = {t0}")
    for name in STEPPERS:
        slope, intercept, r2 = local_slopes[t0][name]
        print(f"    {name}: slope = {slope:.3f},  R^2 = {r2:.5f}")

# Question 1.1 (c)
T = 20.0
global_results = {name: np.empty(len(Ms)) for name in STEPPERS}

for k, M in enumerate(Ms):
    y_true = i_exact(T)
    for name, step_fn in STEPPERS.items():
        t, y = integrate(step_fn, f, i0, T, M)
        global_results[name][k] = abs(y[-1] - y_true)

global_slopes = {}
for name in STEPPERS:
    eps = global_results[name]
    res = linregress(np.log(hs), np.log(eps))
    global_slopes[name] = (res.slope, res.intercept, res.rvalue**2)

# plot
plt.figure(figsize=(6, 5))
for name in STEPPERS:
    eps = global_results[name]
    slope, intercept, r2 = global_slopes[name]
    plt.loglog(hs, eps, "o-", color=colors[name], label=f"{name} (slope={slope:.2f})")
plt.xlabel("h")
plt.ylabel(r"$\varepsilon(h) = |i_M - i(T)|$")
plt.title(f"Global error at T={T}")
plt.grid(True, which="both", alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig("global_error.png", dpi=150)
plt.show()

print("\nQuestion 1.1(c): Global error slopes")
for name in STEPPERS:
    slope, intercept, r2 = global_slopes[name]
    print(f"    {name}: slope = {slope:.3f},  R^2 = {r2:.5f}")
